# Object Detection
Using the MMdetection3D package, we have a wide range of object detection networks we can use.
For our case on incomplete partial rgb scans, we use votenet

In [ ]:
import os
import sys
sys.path.insert(0, '../')
import drm

## File Loading
Read the preprocessed Bin file, the pano matrix and pano image

In [ ]:
binPath = "/home/jvermandere/projects/DRM/_input/Office2.bin"

import trimesh
pcd = drm.load_bin_pointcloud(str(binPath))
trimesh.Scene(pcd).show()


## Object detection
Object detection is performed using the scripts provided by MMdet3D. using the following example command:

```
python  demo/pcd_demo.py 
        demo/data/sunrgbd/000017.bin 
        configs/votenet/votenet_8xb16_sunrgbd-3d.py 
        "/home/jvermandere/projects/DRM/checkpoints/votenet_16x8_sunrgbd-3d-10class_20210820_162823-bf11f014.pth"
```

In [ ]:
detectionJsonPath = drm.detect_objects(str(binPath))

### Detection visualisation

In [ ]:
import trimesh
import numpy as np
import json

with open(detectionJsonPath) as f:
    data = json.load(f)

# Parameters
score_threshold = 0.5  # Only show boxes with score > threshold

# Colormap for labels
label_colors = [
    [1, 0, 0, 0.5],  # red, alpha 0.5
    [0, 1, 0, 0.5],  # green
    [0, 0, 1, 0.5],  # blue
    [1, 1, 0, 0.5],  # yellow
    [1, 0, 1, 0.5],  # magenta
    [0, 1, 1, 0.5],  # cyan
]

# Create list of meshes
bb_meshes = []

for label, score, box in zip(data["labels_3d"], data["scores_3d"], data["bboxes_3d"]):
    if score < score_threshold:
        continue

    center = np.array(box[:3])
    size = np.array(box[3:6])
    rotation_z = box[6]
    color = label_colors[label % len(label_colors)]
    
    mesh = drm.create_trimesh_box(center, size, rotation_z, color)
    bb_meshes.append(mesh)

# Combine meshes for visualization
scene = trimesh.Scene(bb_meshes)
pcd = drm.load_bin_pointcloud(str(binPath))
scene.add_geometry(pcd)

# Show interactive visualization
scene.show()

## Object cut out

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation as R
import os

def points_in_bottom_center_box(points, bottom_center, size, rotation_z, expand_ratio= 1.1):
    """
    Returns a boolean mask of points inside a bottom-center aligned 3D bounding box.
    points: Nx6 array (x,y,z,r,g,b)
    bottom_center: (cx, cy, cz) at the bottom of the box
    size: (l, w, h)
    rotation_z: rotation around z-axis in radians
    """
    # Convert bottom center to geometric center
    center = np.array(bottom_center) + np.array([0, 0, size[2]/2])
    # scale the size after the center is coputed for equal expansion in all directions
    size *= expand_ratio
    
    # Translate points to box frame
    points_local = points[:, :3] - center
    
    # Rotate points by -rotation_z to align with box axes
    rot = R.from_euler('z', -rotation_z).as_matrix()
    points_local = points_local @ rot.T
    
    # Check if points are inside box extents
    mask = (
        (points_local[:, 0] >= -size[0]/2) & (points_local[:, 0] <= size[0]/2) &
        (points_local[:, 1] >= -size[1]/2) & (points_local[:, 1] <= size[1]/2) &
        (points_local[:, 2] >= -size[2]/2) & (points_local[:, 2] <= size[2]/2)
    )
    return mask

def cut_pointcloud_by_bottom_center_boxes(points, labels, scores, boxes, score_threshold=0.5, expand_ratio = 1.1, out_dir="cut_points"):
    """
    Cut a point cloud into separate subsets per bottom-center aligned 3D bounding box.
    Keeps RGB colors intact.

    points: Nx6 array (x,y,z,r,g,b)
    labels: list of int labels
    scores: list of float scores
    boxes: Nx7 array (bottom_center_x, bottom_center_y, bottom_center_z, l, w, h, rotation_z)
    score_threshold: ignore boxes below this score
    out_dir: folder to save each subset
    """
    os.makedirs(out_dir, exist_ok=True)
    points_remaining = points.copy()
    
    # Sort boxes by score descending
    sorted_idx = np.argsort(scores)[::-1]
    labels = [labels[i] for i in sorted_idx]
    scores = [scores[i] for i in sorted_idx]
    boxes = [boxes[i] for i in sorted_idx]
    
    saved_files = []
    
    for i, (label, score, box) in enumerate(zip(labels, scores, boxes)):
        if score < score_threshold:
            continue
        
        bottom_center = np.array(box[:3])
        size = np.array(box[3:6])
        rotation_z = box[6]
        
        mask = points_in_bottom_center_box(points_remaining, bottom_center, size, rotation_z, expand_ratio)
        points_in = points_remaining[mask]
        
        if points_in.shape[0] == 0:
            continue
        
        # Save points inside this box (with RGB)
        filename = os.path.join(out_dir, f"box_{i}_label{label}.txt")
        np.savetxt(filename, points_in, fmt='%.6f')
        saved_files.append(filename)
        
        # Remove these points from remaining
        points_remaining = points_remaining[~mask]
    
    # Save remaining points as isolated (with RGB)
    if points_remaining.shape[0] > 0:
        isolated_file = os.path.join(out_dir, "isolated_points.txt")
        np.savetxt(isolated_file, points_remaining, fmt='%.6f')
        saved_files.append(isolated_file)
    
    print(f"Saved {len(saved_files)} pointclouds in {out_dir}")
    return saved_files

# Example usage:
# points = np.loadtxt("pointcloud.txt")  # Nx6 xyzrgb
# saved_files = cut_pointcloud_by_bottom_center_boxes(points, data["labels_3d"], data["scores_3d"], data["bboxes_3d"])

In [ ]:
# Example usage:
saved_files = cut_pointcloud_by_bottom_center_boxes(np.fromfile(str(binPath), dtype=np.float32).reshape(-1, 6), data["labels_3d"], data["scores_3d"], data["bboxes_3d"],expand_ratio = 1.2, out_dir="../_output/segmented_points")

In [ ]:
pointsColors = np.loadtxt(saved_files[-1], dtype=np.float32).reshape(-1, 6)
cloud = trimesh.points.PointCloud(pointsColors[:, :3], colors=pointsColors[:, 3:6]/255)
scene = trimesh.Scene(cloud)
scene.show()

## Generate Images of detected objects
The detected objects are cropped by their bounding boxes, now we use the bounding box's coordinates to crop the pano image into a pinhole camera

### File Loading

In [ ]:
import numpy as np
from PIL import Image
panoPath = binPath[:-4] + ".png"
panoMatrixPath = binPath[:-4] + "_panomatrix.npy"
panoMatrix = np.load(panoMatrixPath)
print(panoMatrix)
# --- Load equirectangular image ---
pano = Image.open(panoPath).convert("RGB")
width, height = pano.size
img = np.array(pano)

pano

### Pointcloud Alignment Check

In [ ]:

# Y rotation to align pano
Ry = np.array([
    [ 0, 0, -1, 0],
    [ 0, 1, 0, 0],
    [1, 0, 0, 0],
    [ 0, 0, 0, 1]
])
# Convert the pointcloud to uv coordinate space
uv_coords =  drm.transform_xyz_to_uv(cloud.vertices, Ry @ np.linalg.inv(panoMatrix))
projectImage = img.copy()
for coord in uv_coords:
    u = int(coord[0] * width)
    v = int(coord[1] * height)
    #print(u,v)
    projectImage[min(v,height-1), min(u,width-1)] = [int(255), int(0), int(0)]
Image.fromarray(projectImage)

### BoundingBox Alignment Check

In [ ]:
import json
import sys
sys.path.insert(0, '../')
import drm
import cv2
detectionJsonPath = "/home/jvermandere/projects/DRM/_output/preds/Office2.json"

with open(detectionJsonPath) as f:
    data = json.load(f)

# Parameters
score_threshold = 0.5  # Only show boxes with score > threshold
# Copy the images
bbImage = img.copy()
overlay = bbImage.copy()

i = 0
for bb_mesh in bb_meshes:
    uv_coords = drm.transform_xyz_to_uv(bb_mesh.vertices, Ry @ np.linalg.inv(panoMatrix))
    pts = []
    for coord in uv_coords:
        u = int(coord[0] * width)
        v = int(coord[1] * height)
        #print(u,v)
        bbImage[min(v,height-1), min(u,width-1)] = [int(0), int(255), int(0)]
        pts.append([u, v])
    pts = np.array(pts, dtype=np.int32)
    # Compute convex hull
    hull = cv2.convexHull(pts)
    # Draw filled hull on overlay
    cv2.fillConvexPoly(overlay, hull, color)
    i+=1
    
print(i)
alpha = 0.5
bbImage = cv2.addWeighted(overlay, alpha, bbImage, 1 - alpha, 0)
Image.fromarray(bbImage)

### Boundingbox Image Crop
Create pinhole camera cropouts of the pano image using the bounding boxes

In [ ]:
import cv2
import numpy as np

def polygon_to_pinhole(equi_img, poly_pixels, out_res=(800,600), margin=1.1):
    H, W, _ = equi_img.shape
    
    # Convert polygon vertices to spherical coordinates
    thetas = [(x / W) * 2*np.pi - np.pi for x, y in poly_pixels]
    phis = [np.pi/2 - (y / H) * np.pi for x, y in poly_pixels]
    
    # Convert spherical to 3D Cartesian
    verts_3d = np.array([
        [np.cos(phi)*np.sin(theta), np.sin(phi), np.cos(phi)*np.cos(theta)]
        for theta, phi in zip(thetas, phis)
    ])
    
    # Camera look direction = normalized mean of vertices
    center_dir = verts_3d.mean(axis=0)
    center_dir /= np.linalg.norm(center_dir)
    
    # Rotation matrix: rotate z-axis to center_dir
    def look_at_matrix(forward):
        # Orthonormal basis
        forward = forward / np.linalg.norm(forward)
        tmp = np.array([0,1,0]) if abs(forward[1]) < 0.99 else np.array([1,0,0])
        right = np.cross(tmp, forward)
        right /= np.linalg.norm(right)
        up = np.cross(forward, right)
        return np.stack([right, up, forward], axis=1)
    
    R = look_at_matrix(center_dir)
    
    # Transform all vertices to camera frame
    verts_cam = verts_3d @ R
    x_max, x_min = verts_cam[:,0].max(), verts_cam[:,0].min()
    y_max, y_min = verts_cam[:,1].max(), verts_cam[:,1].min()
    z_max, z_min = verts_cam[:,2].max(), verts_cam[:,2].min()
    
    # Compute FOV to include all points, with margin
    fov_x = 2 * np.arctan(margin * max(abs(x_max), abs(x_min)) / max(z_max, 1e-6))
    fov_y = 2 * np.arctan(margin * max(abs(y_max), abs(y_min)) / max(z_max, 1e-6))
    
    # Use the larger FOV for both directions to keep aspect ratio
    fov = np.rad2deg(max(fov_x, fov_y))
    
    # Pinhole camera intrinsics
    w_out, h_out = out_res
    fx = fy = 0.5 * w_out / np.tan(np.deg2rad(fov)/2)
    cx_out, cy_out = w_out / 2, h_out / 2
    
    # Pixel grid
    xx, yy = np.meshgrid(np.arange(w_out), np.arange(h_out))
    x = (xx - cx_out) / fx
    y = (yy - cy_out) / fy
    z = np.ones_like(x)
    
    dirs = np.stack([x, y, z], axis=-1)
    dirs /= np.linalg.norm(dirs, axis=-1, keepdims=True)
    
    # Rotate directions to point at polygon center
    dirs = dirs @ R.T
    
    # Convert to spherical coordinates for sampling
    lon = np.arctan2(dirs[...,0], dirs[...,2])
    lat = np.arcsin(dirs[...,1])
    
    # Map to equirectangular pixels
    u = ((lon + np.pi) / (2*np.pi) * W).astype(np.float32)
    v = ((np.pi/2 - lat) / np.pi * H).astype(np.float32)
    
    # Sample with OpenCV
    pinhole_img = cv2.remap(equi_img, u, v, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)
    return pinhole_img

pinholeImages = []
out_dir = "../_output/cropped_images"
os.makedirs(out_dir, exist_ok=True)
i = 0
for bb_mesh in bb_meshes:
    uv_coords =  drm.transform_xyz_to_uv(bb_mesh.vertices, Ry @ np.linalg.inv(panoMatrix))
    pts = []
    for coord in uv_coords:
        u = int(coord[0] * width)
        v = int(coord[1] * height)
        #print(u,v)
        pts.append([u, v])
    pts = np.array(pts, dtype=np.int32)
    
    equi_img = cv2.flip(polygon_to_pinhole(img, pts, out_res=(800,600), margin=1.4), 0)  # flip the image
    filename = os.path.join(out_dir, f"box_{i}.png")
    print(cv2.imwrite(filename,cv2.cvtColor(equi_img, cv2.COLOR_RGB2BGR)))
    pinholeImages.append(equi_img)
    i+=1

Image.fromarray(pinholeImages[1])